In [3]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
# allow importing the "backbone" package when the kernel's cwd is notebooks/
sys.path.append(os.path.abspath(".."))
import backbone.custom_metrics.AstroMLmod3 as AstroMLmod
import backbone.custom_metrics.Intrinsic_dimension as id_module
from backbone.data_handle.Test import KNN_accuracy
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import davies_bouldin_score

# Radio (RGZ / MiraBest): TPCF, ID and quality metrics per epoch (BYOL vs DINO)

In [ ]:
import importlib
from sklearn.decomposition import IncrementalPCA
importlib.reload(AstroMLmod)

DATA_DIR = "/idia/projects/camil/Koketso/data_"
PLOTS_DIR = "/idia/projects/camil/Koketso/plots"
epochs = range(0, 100, 5)

# per-epoch radio representations (no supervised radio run): RGZ val (unlabelled) for TPCF/ID/cosine/anisotropy,
# MiraBest FRI/FRII (train + test) for Davies-Bouldin and KNN accuracy
runs = {
    "BYOL": "byol_rgz",
    "DINO": "dino_rgz",
}


def read_representations(path):
    df = pd.read_csv(path)
    return df.drop(columns=["Image_names", "label"]).to_numpy(), df["label"].tolist()


rows = []
for method, prefix in runs.items():
    for epoch in epochs:
        paths = [f"{DATA_DIR}/{prefix}_{split}_representations_epoch_{epoch}.csv" for split in ("val", "train", "knn_test")]
        if not all(os.path.exists(path) for path in paths):
            continue
        representations, _ = read_representations(paths[0])
        train_rep, train_labels = read_representations(paths[1])
        test_rep, test_labels = read_representations(paths[2])
        labelled_rep, labels = np.concatenate([train_rep, test_rep]), train_labels + test_labels

        # TPCF_score bootstraps over subsamples and returns (mean, std)
        tpcf, tpcf_std = AstroMLmod.TPCF_score(representations, epoch=epoch)
        id_mean, id_std = id_module.id_score(representations)
        db_score = davies_bouldin_score(labelled_rep, labels)
        knn_acc, knn_acc_std = KNN_accuracy(labelled_rep, labels)

        # average pairwise cosine similarity over off-diagonal pairs, from the unit-vector sum and Gram matrix
        # (avoids building the n x n similarity matrix for ~20k RGZ sources)
        unit_representations = representations / np.linalg.norm(representations, axis=1, keepdims=True)
        n = len(unit_representations)
        unit_sum = unit_representations.sum(axis=0, dtype=np.float64)
        gram = (unit_representations.T @ unit_representations).astype(np.float64)
        avg_cos_sim = (unit_sum @ unit_sum - n) / (n * (n - 1))
        std_cos_sim = np.sqrt(((gram ** 2).sum() - n) / (n * (n - 1)) - avg_cos_sim ** 2)

        # anisotropy: fraction of total variance explained by the top principal component of the unit vectors
        anisotropy = IncrementalPCA(n_components=1).fit(unit_representations).explained_variance_ratio_[0]

        rows.append({
            "method": method,
            "epoch": epoch,
            "tpcf_score": tpcf,
            "tpcf_std": tpcf_std,
            "id_mean": float(np.ravel(id_mean)[0]),
            "id_std": float(np.ravel(id_std)[0]),
            "avg_cosine_similarity": avg_cos_sim,
            "cosine_similarity_std": std_cos_sim,
            "anisotropy": anisotropy,
            "davies_bouldin": db_score,
            "knn_accuracy": knn_acc,
            "knn_accuracy_std": knn_acc_std,
        })
        print(f"{method} epoch {epoch}: TPCF={tpcf:.4f}±{tpcf_std:.4f} ID={rows[-1]['id_mean']:.2f} CosineSim={avg_cos_sim:.4f}±{std_cos_sim:.4f} Anisotropy={anisotropy:.4f} DaviesBouldin={db_score:.4f} KNNAcc={knn_acc:.2f}±{knn_acc_std:.2f}")

BYOL epoch 0: TPCF=126.8300±3.5700 ID=23.99 CosineSim=0.4101±0.1928 Anisotropy=0.2014 DaviesBouldin=5.8930 KNNAcc=58.33±0.84


In [ ]:
results = pd.DataFrame(rows)
results.to_csv(f"{PLOTS_DIR}/radio_metrics_per_epoch.csv", index=False)
results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import string

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
})

ACCENT = "#1f4e79"
# fixed categorical order: colour follows the method
COLORS = {"Supervised": "#2a78d6", "BYOL": "#eb6834", "DINO": "#1baf7a"}

metrics = [
    ("tpcf_score", "TPCF score", "tpcf_std"),
    ("id_mean", "Intrinsic dimension", "id_std"),
    ("avg_cosine_similarity", "Avg. cosine similarity", "cosine_similarity_std"),
    ("anisotropy", "Anisotropy (top-1 EVR)", None),
    ("davies_bouldin", "Davies-Bouldin score", None),
    ("knn_accuracy", "KNN accuracy (%)", "knn_accuracy_std"),
]


def plot_gradient_band(ax, x, y, y_err, color=ACCENT, n_layers=40, max_sigma=3.0):
    """Shade the uncertainty band with an alpha gradient that fades away from the mean."""
    x = np.asarray(x)
    y = np.asarray(y)
    y_err = np.asarray(y_err)
    sigmas = np.linspace(max_sigma / n_layers, max_sigma, n_layers)
    for sigma in sigmas[::-1]:
        alpha = 0.35 * np.exp(-0.5 * sigma ** 2)
        ax.fill_between(x, y - sigma * y_err, y + sigma * y_err, color=color, alpha=alpha, linewidth=0)


n_cols = 3
n_rows = -(-len(metrics) // n_cols)  # ceil division

fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(3.3 * n_cols, 2.5 * n_rows), squeeze=False, constrained_layout=True
)
axes = axes.flatten()

for panel_label, ax, (col, title, err_col) in zip(string.ascii_lowercase, axes, metrics):
    for method, data in results.groupby("method", sort=False):
        color = COLORS[method]
        ax.plot(data["epoch"], data[col], color=color, label=method)
        if col == "tpcf_score":
            plot_gradient_band(ax, data["epoch"], data[col], data[err_col], color=color)
        elif err_col is not None:
            ax.fill_between(
                data["epoch"],
                data[col] - data[err_col],
                data[col] + data[err_col],
                color=color,
                alpha=0.2,
            )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.text(-0.15, 1.08, f"({panel_label})", transform=ax.transAxes, fontsize=11, fontweight="bold")

fig.legend(*axes[0].get_legend_handles_labels(), loc="outside upper center", ncol=len(COLORS))

# hide any unused axes in the grid
for ax in axes[len(metrics):]:
    ax.axis("off")

fig.savefig(f"{PLOTS_DIR}/radio_metrics_per_epoch_grid.pdf", bbox_inches="tight")
fig.savefig(f"{PLOTS_DIR}/radio_metrics_per_epoch_grid.png", bbox_inches="tight")
plt.show()